### 1. Kết nối runtime chế độ GPU

### 2. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Clone Mã Nguồn

In [2]:
%cd /content
!git clone https://github.com/Violetta147/face-recognition-depth-pad.git
%cd /content/face-recognition-depth-pad

/content
Cloning into 'face-recognition-depth-pad'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 108 (delta 8), reused 106 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 153.21 KiB | 2.69 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/face-recognition-depth-pad


### Cài project

In [3]:
!python -m pip install -q -e ".[dev]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for deepface-pad (pyproject.toml) ... done


### Chạy test

In [4]:
!pytest -q

...............................                                          [100%]
31 passed in 11.73s


### 3. Giải nén dataset vào bộ nhớ Colab

In [6]:
!mkdir -p /content/datasets
!unzip -q "/content/drive/MyDrive/face-pad/casia-fasd.zip" -d /content/datasets

### Kiểm tra

In [7]:
!find /content/datasets/casia-fasd -maxdepth 2 -type d

/content/datasets/casia-fasd
/content/datasets/casia-fasd/train
/content/datasets/casia-fasd/train/live
/content/datasets/casia-fasd/train/spoof
/content/datasets/casia-fasd/test
/content/datasets/casia-fasd/test/live
/content/datasets/casia-fasd/test/spoof


### 4. Tạo manifest trên Colab

In [8]:
!python scripts/prepare_casia_fasd.py \
  --data-root /content/datasets/casia-fasd \
  --frames-per-video 20

             sample_id  video_id
split label                     
test  0           5400       270
      1           1800        90
train 0           2880       144
      1            960        48
val   0            720        36
      1            240        12
wrote 12000 sampled frames from 600 videos


### Kiểm tra leakage lần nữa

In [9]:
!python scripts/validate_manifest.py \
  data/manifests/casia_fasd_debug.csv \
  --data-root /content/datasets/casia-fasd \
  --subject-disjoint

Manifest valid


### 5. Tạo cấu hình runtime

In [10]:
from pathlib import Path
import yaml

source = Path("configs/casia_e0_smoke.yaml")
config = yaml.safe_load(source.read_text())

config["data"]["root"] = "/content/datasets/casia-fasd"
config["runs_dir"] = "/content/drive/MyDrive/face-pad/runs"

runtime_config = Path("/content/casia_e0_smoke.yaml")
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))

print(runtime_config.read_text())

experiment_id: CASIA_E0_SMOKE
seed: 42
runs_dir: /content/drive/MyDrive/face-pad/runs
model:
  name: mobilenet_v3_small
  pretrained: true
data:
  manifest: data/manifests/casia_fasd_debug.csv
  root: /content/datasets/casia-fasd
  image_size: 224
loss:
  classification: bce
  lambda_cls: 1.0
training:
  epochs: 1
  lr: 0.0003
  batch_size: 32
  workers: 2
  weight_decay: 0.0001
evaluation:
  aggregation: mean



### 6. Chuyển sang GPU

In [11]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA: True
GPU: NVIDIA A100-SXM4-40GB


### 7. Chạy E0 smoke test một epoch

In [12]:
%cd /content/face-recognition-depth-pad
!python scripts/run_experiment.py /content/casia_e0_smoke.yaml

/content/face-recognition-depth-pad
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100% 9.83M/9.83M [00:00<00:00, 108MB/s]
/content/drive/MyDrive/face-pad/runs/CASIA_E0_SMOKE_seed42_20260920T081731Z


### 8. Kiểm tra kết quả

In [13]:
from pathlib import Path
import json

runs = Path("/content/drive/MyDrive/face-pad/runs")
latest = max(runs.iterdir(), key=lambda path: path.stat().st_mtime)

print("Latest run:", latest)
print(json.loads((latest / "metrics.json").read_text()))

Latest run: /content/drive/MyDrive/face-pad/runs/CASIA_E0_SMOKE_seed42_20260920T081731Z
{'threshold': 3.986435517617792e-05, 'apcer': 0.0, 'bpcer': 0.0, 'acer': 0.0, 'eer': 0.0, 'auc': 1.0}


### Chạy cell sau để kiểm tra phân phối score

In [14]:
from pathlib import Path
import pandas as pd
import json

run = Path(
    "/content/drive/MyDrive/face-pad/runs/"
    "CASIA_E0_SMOKE_seed42_20260920T081731Z"
)

print("Training log")
display(pd.read_csv(run / "train_log.csv"))

frame_scores = pd.read_csv(run / "val_frame_scores.csv")
video_scores = pd.read_csv(run / "val_scores.csv")

print("Frame-level score distribution")
display(
    frame_scores.groupby("label")["score"]
    .describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99])
)

print("Video-level score distribution")
display(
    video_scores.groupby("label")["score"]
    .describe(percentiles=[0.01, 0.1, 0.5, 0.9, 0.99])
)

print("Metrics")
print(json.loads((run / "metrics.json").read_text()))

Training log


,stage,epoch,train_loss,val_loss
0,joint,1,0.070382,1.495542


Frame-level score distribution


,count,mean,std,min,1%,10%,50%,90%,99%,max
label,,,,,,,,,,
0,720.0,0.000004,0.000008,1.799878e-08,4.022361e-08,1.154029e-07,0.000002,0.000009,0.000043,0.000098
1,240.0,0.118645,0.229394,3.163610e-06,4.912424e-06,3.408675e-05,0.001279,0.497835,0.835151,0.941224


Video-level score distribution


,count,mean,std,min,1%,10%,50%,90%,99%,max
label,,,,,,,,,,
0,36.0,0.000004,0.000005,7.749003e-08,7.818916e-08,2.021361e-07,0.000003,0.000009,0.000019,0.000020
1,12.0,0.118645,0.210005,3.986436e-05,4.938689e-05,1.330825e-04,0.001753,0.440748,0.579350,0.594492


Metrics
{'threshold': 3.986435517617792e-05, 'apcer': 0.0, 'bpcer': 0.0, 'acer': 0.0, 'eer': 0.0, 'auc': 1.0}


### Kiểm tra khoảng cách giữa hai lớp

In [15]:
attacks = video_scores.loc[video_scores.label == 0, "score"]
live = video_scores.loc[video_scores.label == 1, "score"]

print("Highest attack score:", attacks.max())
print("Lowest live score:", live.min())
print("Gap:", live.min() - attacks.max())

Highest attack score: 1.9691866100401967e-05
Lowest live score: 3.986435517617792e-05
Gap: 2.0172489075775954e-05


### Xác định các video biên

In [16]:
print("Hardest attacks")
display(
    video_scores[video_scores.label == 0]
    .nlargest(5, "score")
)

print("Hardest live")
display(
    video_scores[video_scores.label == 1]
    .nsmallest(5, "score")
)

print("Hardest live frames")
display(
    frame_scores[frame_scores.label == 1]
    .nsmallest(10, "score")
)

print("Hardest attack frames")
display(
    frame_scores[frame_scores.label == 0]
    .nlargest(10, "score")
)

Hardest attacks


,video_id,label,score
27,casia_s14_v4,0,0.000020
39,casia_s19_v4,0,0.000017
19,casia_s09_v8,0,0.000013
18,casia_s09_v7,0,0.000009
22,casia_s09_vHR_3,0,0.000009


Hardest live


,video_id,label,score
1,casia_s04_v2,1,0.000040
8,casia_s04_vHR_1,1,0.000126
0,casia_s04_v1,1,0.000193
12,casia_s09_v1,1,0.000198
20,casia_s09_vHR_1,1,0.000558


Hardest live frames


,sample_id,video_id,label,score
21,casia_s04_v2_f000010,casia_s04_v2,1,0.000003
30,casia_s04_v2_f000105,casia_s04_v2,1,0.000003
255,casia_s09_v1_f000145,casia_s09_v1,1,0.000005
25,casia_s04_v2_f000052,casia_s04_v2,1,0.000005
29,casia_s04_v2_f000094,casia_s04_v2,1,0.000007
26,casia_s04_v2_f000063,casia_s04_v2,1,0.000007
240,casia_s09_v1_f000000,casia_s09_v1,1,0.000007
23,casia_s04_v2_f000031,casia_s04_v2,1,0.000008
33,casia_s04_v2_f000136,casia_s04_v2,1,0.000010
28,casia_s04_v2_f000084,casia_s04_v2,1,0.000011


Hardest attack frames


,sample_id,video_id,label,score
551,casia_s14_v4_f000113,casia_s14_v4,0,0.000098
387,casia_s09_v8_f000053,casia_s09_v8,0,0.000055
63,casia_s04_v4_f000031,casia_s04_v4,0,0.000053
554,casia_s14_v4_f000144,casia_s14_v4,0,0.000052
550,casia_s14_v4_f000103,casia_s14_v4,0,0.000051
783,casia_s19_v4_f000031,casia_s19_v4,0,0.000046
380,casia_s09_v8_f000000,casia_s09_v8,0,0.000044
798,casia_s19_v4_f000185,casia_s19_v4,0,0.000043
552,casia_s14_v4_f000124,casia_s14_v4,0,0.000042
390,casia_s09_v8_f000076,casia_s09_v8,0,0.000041


### Tạo config mới

In [17]:
from pathlib import Path
import yaml

source = Path("/content/casia_e0_smoke.yaml")
config = yaml.safe_load(source.read_text())

config["experiment_id"] = "CASIA_E0_BCE_5E"
config["training"]["epochs"] = 5

output = Path("/content/casia_e0_bce_5e.yaml")
output.write_text(yaml.safe_dump(config, sort_keys=False))

print(output.read_text())

experiment_id: CASIA_E0_BCE_5E
seed: 42
runs_dir: /content/drive/MyDrive/face-pad/runs
model:
  name: mobilenet_v3_small
  pretrained: true
data:
  manifest: data/manifests/casia_fasd_debug.csv
  root: /content/datasets/casia-fasd
  image_size: 224
loss:
  classification: bce
  lambda_cls: 1.0
training:
  epochs: 5
  lr: 0.0003
  batch_size: 32
  workers: 2
  weight_decay: 0.0001
evaluation:
  aggregation: mean



In [18]:
%cd /content/face-recognition-depth-pad
!python scripts/run_experiment.py /content/casia_e0_bce_5e.yaml

/content/face-recognition-depth-pad
/content/drive/MyDrive/face-pad/runs/CASIA_E0_BCE_5E_seed42_20260920T082941Z


In [19]:
from pathlib import Path
import pandas as pd
import json

runs = Path("/content/drive/MyDrive/face-pad/runs")
run = max(
    (path for path in runs.iterdir() if path.name.startswith("CASIA_E0_BCE_5E")),
    key=lambda path: path.stat().st_mtime
)

print("Run:", run)
display(pd.read_csv(run / "train_log.csv"))
print(json.loads((run / "metrics.json").read_text()))

frame_scores = pd.read_csv(run / "val_frame_scores.csv")
video_scores = pd.read_csv(run / "val_scores.csv")

display(frame_scores.groupby("label")["score"].describe())
display(video_scores.groupby("label")["score"].describe())

attacks = video_scores.loc[video_scores.label == 0, "score"]
live = video_scores.loc[video_scores.label == 1, "score"]

print("Highest attack:", attacks.max())
print("Lowest live:", live.min())
print("Gap:", live.min() - attacks.max())

Run: /content/drive/MyDrive/face-pad/runs/CASIA_E0_BCE_5E_seed42_20260920T082941Z


,stage,epoch,train_loss,val_loss
0,joint,1,0.070382,1.495542
1,joint,2,0.005806,0.480405
2,joint,3,0.001817,0.028079
3,joint,4,0.006268,0.152174
4,joint,5,0.006099,0.002806


{'threshold': 0.9900876432657242, 'apcer': 0.0, 'bpcer': 0.0, 'acer': 0.0, 'eer': 0.0, 'auc': 1.0}


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,720.0,0.001901,0.032593,3.799946e-13,1.084301e-09,1.248844e-07,0.000006,0.768765
1,240.0,0.997922,0.007251,8.889989e-01,9.973770e-01,9.990065e-01,0.999644,0.999882


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,36.0,0.001901,0.010012,1.827997e-10,1.380871e-08,9.931462e-07,0.000044,0.060134
1,12.0,0.997922,0.002690,9.900876e-01,9.973870e-01,9.988160e-01,0.999596,0.999845


Highest attack: 0.0601338233259866
Lowest live: 0.9900876432657242
Gap: 0.9299538199397376


### Chỉnh lỗi python import package blah blah

In [21]:
from pathlib import Path

print("Project:", Path("/content/face-recognition-depth-pad").exists())
print("Dataset:", Path("/content/datasets/casia-fasd").exists())
print(
    "Manifest:",
    Path(
        "/content/face-recognition-depth-pad/"
        "data/manifests/casia_fasd_debug.csv"
    ).exists(),
)

Project: True
Dataset: True
Manifest: True


In [22]:
%cd /content/face-recognition-depth-pad
%pip install -q -e .

/content/face-recognition-depth-pad
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for deepface-pad (pyproject.toml) ... done


In [23]:
import deepface_pad
print(deepface_pad.__version__)

ModuleNotFoundError: No module named 'deepface_pad'

In [24]:
import sys
from pathlib import Path

project_root = Path("/content/face-recognition-depth-pad")
source_root = project_root / "src"

if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

import deepface_pad

print("Package:", deepface_pad.__version__)
print("Source:", deepface_pad.__file__)

Package: 0.1.0
Source: /content/face-recognition-depth-pad/src/deepface_pad/__init__.py


In [25]:
from deepface_pad.data import PadDataset
from deepface_pad.metrics import aggregate_video_scores, evaluate_scores
from deepface_pad.train import build_model, score_loader

print("Imports successful")

Imports successful


### Sử dụng threshold đã khóa từ validation và không chọn lại threshold trên test

In [26]:
from pathlib import Path
import json
import yaml
import torch
import pandas as pd

from torch.utils.data import DataLoader

from deepface_pad.data import PadDataset
from deepface_pad.metrics import aggregate_video_scores, evaluate_scores
from deepface_pad.train import build_model, score_loader

run = Path(
    "/content/drive/MyDrive/face-pad/runs/"
    "CASIA_E0_BCE_5E_seed42_20260920T082941Z"
)

config = yaml.safe_load((run / "config.yaml").read_text())
threshold = json.loads((run / "threshold.json").read_text())["threshold"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_model(config).to(device)
checkpoint = torch.load(
    run / "best.ckpt",
    map_location=device,
    weights_only=False,
)
model.load_state_dict(checkpoint["model"])

test_dataset = PadDataset(
    manifest=config["data"]["manifest"],
    data_root=config["data"]["root"],
    split="test",
    image_size=config["data"]["image_size"],
    augment=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config["training"]["batch_size"],
    shuffle=False,
    num_workers=config["training"]["workers"],
)

frame_scores = score_loader(model, test_loader, device)
video_scores = aggregate_video_scores(
    frame_scores,
    config["evaluation"]["aggregation"],
)

metrics = evaluate_scores(
    video_scores["label"].to_numpy(),
    video_scores["score"].to_numpy(),
    threshold=threshold,
)

frame_scores.to_csv(run / "test_frame_scores.csv", index=False)
video_scores.to_csv(run / "test_scores.csv", index=False)
(run / "test_metrics.json").write_text(
    json.dumps(metrics.to_dict(), indent=2)
)

print("Locked validation threshold:", threshold)
print("Test metrics:", metrics.to_dict())

display(frame_scores.groupby("label")["score"].describe())
display(video_scores.groupby("label")["score"].describe())

attacks = video_scores.loc[video_scores.label == 0, "score"]
live = video_scores.loc[video_scores.label == 1, "score"]

print("Highest test attack:", attacks.max())
print("Lowest test live:", live.min())
print("Test gap:", live.min() - attacks.max())

Locked validation threshold: 0.9900876432657242
Test metrics: {'threshold': 0.9900876432657242, 'apcer': 0.0, 'bpcer': 0.1111111111111111, 'acer': 0.05555555555555555, 'eer': 0.024074074074074074, 'auc': 0.9979835390946502}


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,5400.0,0.013123,0.095721,2.036846e-16,4.416018e-10,4.910434e-08,0.000011,0.983478
1,1800.0,0.963497,0.173315,5.803994e-05,9.959089e-01,9.981526e-01,0.999293,0.999862


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,270.0,0.013123,0.090888,1.749461e-13,2.852945e-09,5.599957e-07,0.000077,0.946375
1,90.0,0.963497,0.172781,1.906037e-03,9.959031e-01,9.979727e-01,0.999247,0.999743


Highest test attack: 0.9463753312826156
Lowest test live: 0.0019060372604144504
Test gap: -0.9444692940222011


### Sửa DataFrame

In [28]:
video_scores = video_scores.reset_index(drop=True)

### Error Analysis

In [29]:
manifest = pd.read_csv(config["data"]["manifest"])

video_metadata = (
    manifest[
        ["video_id", "subject_id", "label", "attack_type", "quality"]
    ]
    .drop_duplicates("video_id")
    .reset_index(drop=True)
)

analysis = video_scores.merge(
    video_metadata,
    on=["video_id", "label"],
    how="left",
)

analysis["prediction"] = (
    analysis["score"] >= threshold
).astype(int)

analysis["correct"] = (
    analysis["prediction"] == analysis["label"]
)

display(
    analysis.groupby(
        ["label", "attack_type", "quality"],
        dropna=False,
    ).agg(
        videos=("video_id", "count"),
        errors=("correct", lambda values: (~values).sum()),
        mean_score=("score", "mean"),
        min_score=("score", "min"),
        max_score=("score", "max"),
    )
)

false_rejects = (
    analysis[
        (analysis["label"] == 1) &
        (~analysis["correct"])
    ]
    .sort_values("score")
)

print("Misclassified bona fide videos")
display(false_rejects)

print("Highest-scoring attacks")
display(
    analysis[analysis["label"] == 0]
    .nlargest(15, "score")
)

analysis.to_csv(run / "test_error_analysis.csv", index=False)
false_rejects.to_csv(run / "test_false_rejects.csv", index=False)

videos  errors  mean_score     min_score  \
label attack_type  quality                                             
0     cut_photo    high         30       0    0.000617  1.749461e-13   
                   low          30       0    0.001280  6.470574e-11   
                   normal       30       0    0.000612  5.332273e-13   
      video_replay high         30       0    0.050555  8.330194e-12   
                   low          30       0    0.005346  4.805573e-12   
                   normal       30       0    0.035518  5.516315e-12   
      warped_photo high         30       0    0.004587  4.163838e-12   
                   low          30       0    0.003688  3.158259e-10   
                   normal       30       0    0.015900  2.052839e-13   
1     live         high         30       3    0.994928  9.746980e-01   
                   low          30       3    0.933257  1.906037e-03   
                   normal       30       4    0.962304  6.655947e-02   

                            max_score  
label attack_type  quality             
0     cut_photo    high      0.014714  
                   low       0.024846  
                   normal    0.005877  
      video_replay high      0.946375  
                   low       0.119147  
                   normal    0.937595  
      warped_photo high      0.072221  
                   low       0.050587  
                   normal    0.370825  
1     live         high      0.998747  
                   low       0.999743  
                   normal    0.999739

Misclassified bona fide videos


,video_id,label,score,subject_id,attack_type,quality,prediction,correct
25,casia_s23_v2,1,0.001906,casia_s23,live,low,0,False
109,casia_s30_v2,1,0.051869,casia_s30,live,low,0,False
24,casia_s23_v1,1,0.066559,casia_s23,live,normal,0,False
12,casia_s22_v1,1,0.931139,casia_s22,live,normal,0,False
216,casia_s39_v1,1,0.946829,casia_s39,live,normal,0,False
108,casia_s30_v1,1,0.969971,casia_s30,live,normal,0,False
8,casia_s21_vHR_1,1,0.974698,casia_s21,live,high,0,False
128,casia_s31_vHR_1,1,0.982409,casia_s31,live,high,0,False
325,casia_s48_v2,1,0.989456,casia_s48,live,low,0,False
224,casia_s39_vHR_1,1,0.989988,casia_s39,live,high,0,False


Highest-scoring attacks


,video_id,label,score,subject_id,attack_type,quality,prediction,correct
11,casia_s21_vHR_4,0,0.946375,casia_s21,video_replay,high,0,True
6,casia_s21_v7,0,0.937595,casia_s21,video_replay,normal,0,True
203,casia_s37_vHR_4,0,0.557268,casia_s37,video_replay,high,0,True
278,casia_s44_v3,0,0.370825,casia_s44,warped_photo,normal,0,True
187,casia_s36_v8,0,0.119147,casia_s36,video_replay,low,0,True
174,casia_s35_v7,0,0.109126,casia_s35,video_replay,normal,0,True
9,casia_s21_vHR_2,0,0.072221,casia_s21,warped_photo,high,0,True
2,casia_s21_v3,0,0.061210,casia_s21,warped_photo,normal,0,True
213,casia_s38_vHR_2,0,0.054018,casia_s38,warped_photo,high,0,True
183,casia_s36_v4,0,0.050587,casia_s36,warped_photo,low,0,True


### Kết luận E0 tạm thời


### CASIA-FASD temporary debug benchmark
### Model: MobileNetV3 Small, pretrained
### Seed: 42
### Epochs: 5
### Aggregation: mean
### Threshold source: validation
### Test APCER: 0.00%
### Test BPCER: 11.11%
### Test ACER: 5.56%
### Test EER: 2.41%
### Test AUC: 99.80%